In [1]:
#!/usr/bin/env python3
"""
Clean filtered_raw_sysadmin_comments_new.csv using the EXACT SAME logic as
comment_data_pre-process.ipynb's clean_filtered_comments() function, then
append the result to the existing master_comments_filtered.csv and update
comment_filter_summary.csv.

This does NOT re-run the top-level/target-post filtering step -- that was
already done when you built filtered_raw_sysadmin_comments_new.csv. This
script only applies the cleaning stage (dedup, date window, deleted/removed
removal, word-count filter, text cleaning) so sysadmin is filtered
identically to the other four subreddits, then merges.

Run this from /Users/nadia/Desktop/redditRun_june/comment_data/
(or edit OUTPUT_DIR / INPUT paths below).
"""

import pandas as pd
import re
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG -- match the notebook exactly ────────────────────────────────────
OUTPUT_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'

NEW_SYSADMIN_RAW_FILE = OUTPUT_DIR + 'filtered_raw_sysadmin_comments_new.csv'
MASTER_FILE           = OUTPUT_DIR + 'master_comments_filtered.csv'
SUMMARY_FILE          = OUTPUT_DIR + 'comment_filter_summary.csv'
BAT_SCORES_FILE       = OUTPUT_DIR + 'bat_score_pos.csv'

START_DATE = '2018-01-01'
END_DATE   = '2026-04-26'   # matches notebook exactly
MIN_COMMENT_WORDS = 10

OUT_COLS = ['id', 'post_id', 'parent_id', 'link_id', 'author',
            'created_date', 'created_utc', 'year', 'month', 'year_month',
            'body', 'cleaned_body', 'word_count', 'score', 'permalink',
            'subreddit_source']

EXPECTED_COLS = ['id', 'author', 'body', 'created_utc', 'score',
                  'link_id', 'parent_id', 'permalink', 'subreddit', 'post_id']


# ── TEXT CLEANING -- copied verbatim from the notebook ──────────────────────
def clean_text(text):
    if pd.isna(text) or text == "":
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', ' URL ', text, flags=re.MULTILINE)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'\*\*([^\*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^\*]+)\*', r'\1', text)
    text = re.sub(r'~~([^~]+)~~', r'\1', text)
    text = re.sub(r'#{1,6}\s', '', text)
    text = re.sub(r'r/\w+', ' SUBREDDIT ', text)
    text = re.sub(r'u/\w+', ' USER ', text)
    text = re.sub(r'\[([^\]]*)\]', '', text)
    text = re.sub(r'\(([^\)]*)\)', '', text)
    text = re.sub(r'\{([^\}]*)\}', '', text)
    text = re.sub(r'[^\w\s\.\-_:/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^a-zA-Z\s\.\,\!\?\;\:]', ' ', text)
    text = text.lower()
    return text


# ── CLEANING -- copied verbatim from the notebook's clean_filtered_comments()
def clean_filtered_comments(comments):
    if comments.empty:
        return comments

    before = len(comments)
    comments = comments.drop_duplicates(subset=['id'])
    print(f"  Removed {before - len(comments):,} duplicate comments")

    comments['created_date'] = pd.to_datetime(comments['created_utc'], unit='s', errors='coerce')
    comments['year']       = comments['created_date'].dt.year
    comments['month']      = comments['created_date'].dt.month
    comments['year_month'] = comments['created_date'].dt.to_period('M').astype(str)

    before = len(comments)
    comments = comments[comments['created_date'].notna()]
    comments = comments[
        (comments['created_date'] >= START_DATE) &
        (comments['created_date'] <= END_DATE)
    ]
    print(f"  Date window filter: kept {len(comments):,} (removed {before - len(comments):,})")

    comments['body'] = comments['body'].astype(str)
    before = len(comments)
    comments = comments[~comments['body'].isin(['[deleted]', '[removed]', 'nan', ''])]
    print(f"  Removed {before - len(comments):,} deleted/removed comments")

    comments['cleaned_body'] = comments['body'].apply(clean_text)
    comments['word_count']   = comments['cleaned_body'].str.split().str.len()

    before = len(comments)
    comments = comments[comments['word_count'] >= MIN_COMMENT_WORDS]
    print(f"  Removed {before - len(comments):,} comments under {MIN_COMMENT_WORDS} words")

    before = len(comments)
    comments = comments[comments['cleaned_body'].str.strip().str.len() > 0]
    print(f"  Removed {before - len(comments):,} comments with empty cleaned text")

    return comments


def main():
    print("=" * 80)
    print("CLEAN + MERGE NEW SYSADMIN COMMENTS INTO MASTER FILE")
    print("=" * 80)

    # --- Load new sysadmin raw-filtered file ---
    if not os.path.exists(NEW_SYSADMIN_RAW_FILE):
        print(f"[ERROR] Not found: {NEW_SYSADMIN_RAW_FILE}")
        sys.exit(1)

    df = pd.read_csv(NEW_SYSADMIN_RAW_FILE)
    print(f"\nLoaded {NEW_SYSADMIN_RAW_FILE}: {len(df):,} rows")
    print(f"Columns present: {list(df.columns)}")

    # --- Column sanity check against notebook's expected schema ---
    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing:
        print(f"\n[WARN] Missing expected columns: {missing}")
        print("       Update the column-mapping block below before proceeding,")
        print("       or the cleaning step will fail (needs at least: id, body, created_utc).")
        # Common rename map in case your sysadmin filter script used different names --
        # EDIT this dict if your actual headers differ.
        rename_map = {
            'comment_id': 'id',
            'text': 'body',
            'timestamp': 'created_utc',
            'target_post_id': 'post_id',
        }
        applicable = {k: v for k, v in rename_map.items() if k in df.columns and v not in df.columns}
        if applicable:
            print(f"       Auto-applying rename: {applicable}")
            df = df.rename(columns=applicable)
        missing_after = [c for c in ['id', 'body', 'created_utc', 'post_id'] if c not in df.columns]
        if missing_after:
            print(f"[ERROR] Still missing required columns after rename attempt: {missing_after}")
            print("        Cannot proceed safely -- fix column names manually.")
            sys.exit(1)

    if 'subreddit' not in df.columns:
        df['subreddit'] = 'sysadmin'

    # --- Verify post_ids actually match bat_score>0 target set (sanity check) ---
    if os.path.exists(BAT_SCORES_FILE):
        bat = pd.read_csv(BAT_SCORES_FILE, usecols=['post_id', 'bat_score'])
        target_ids = set(bat.loc[bat['bat_score'] > 0, 'post_id'].astype(str))
        df['post_id'] = df['post_id'].astype(str)
        n_valid = df['post_id'].isin(target_ids).sum()
        n_total = len(df)
        if n_valid < n_total:
            print(f"\n[WARN] {n_total - n_valid:,} of {n_total:,} rows have post_id NOT in "
                  f"bat_score>0 target set -- dropping these (they shouldn't be here).")
            df = df[df['post_id'].isin(target_ids)]
        else:
            print(f"\n  post_id check OK: all {n_total:,} rows match bat_score>0 target posts")

    # --- Apply identical cleaning pipeline ---
    print(f"\nCleaning r/sysadmin comments (same logic as notebook)…")
    cleaned = clean_filtered_comments(df)
    cleaned['subreddit_source'] = 'sysadmin'

    cleaned = cleaned[[c for c in OUT_COLS if c in cleaned.columns]]
    print(f"\n  r/sysadmin final cleaned count: {len(cleaned):,}")

    # --- Load existing master file ---
    if not os.path.exists(MASTER_FILE):
        print(f"[ERROR] Master file not found: {MASTER_FILE}")
        sys.exit(1)

    master = pd.read_csv(MASTER_FILE)
    print(f"\nExisting master_comments_filtered.csv: {len(master):,} rows")

    if 'subreddit_source' in master.columns and (master['subreddit_source'] == 'sysadmin').any():
        n_old_sysadmin = (master['subreddit_source'] == 'sysadmin').sum()
        print(f"[WARN] Master file already contains {n_old_sysadmin:,} rows with "
              f"subreddit_source == 'sysadmin' (the old truncated data).")
        print("       Dropping these before merging in the new sysadmin data, to avoid duplicates.")
        master = master[master['subreddit_source'] != 'sysadmin']

    # --- Duplicate check across id ---
    combined = pd.concat([master, cleaned], ignore_index=True)
    before = len(combined)
    dupe_mask = combined.duplicated(subset=['id'], keep='first')
    n_dupes = dupe_mask.sum()
    if n_dupes:
        print(f"\n[WARN] {n_dupes:,} duplicate comment 'id' values found across master + new sysadmin "
              f"-- dropping (keep first occurrence)")
        combined = combined[~dupe_mask]
    else:
        print(f"\n  No duplicate comment ids between master and new sysadmin data.")

    combined = combined[[c for c in OUT_COLS if c in combined.columns]]
    combined.to_csv(MASTER_FILE, index=False)
    print(f"\n✓ Updated {MASTER_FILE}  →  {len(combined):,} total comments "
          f"(was {before - n_dupes - len(cleaned) + len(cleaned):,}... net {len(combined):,})")

    # --- Update summary file ---
    if os.path.exists(SUMMARY_FILE):
        summary = pd.read_csv(SUMMARY_FILE)
        summary = summary[summary['subreddit'] != 'sysadmin']
    else:
        summary = pd.DataFrame(columns=['subreddit', 'comments_kept'])

    summary = pd.concat([summary, pd.DataFrame([{
        'subreddit': 'sysadmin', 'comments_kept': len(cleaned)
    }])], ignore_index=True)
    summary.to_csv(SUMMARY_FILE, index=False)
    print(f"✓ Updated {SUMMARY_FILE}")

    # --- Final coverage stats ---
    if os.path.exists(BAT_SCORES_FILE):
        bat = pd.read_csv(BAT_SCORES_FILE, usecols=['post_id', 'bat_score'])
        target_ids = set(bat.loc[bat['bat_score'] > 0, 'post_id'].astype(str))
        combined['post_id'] = combined['post_id'].astype(str)
        covered = combined['post_id'].nunique()
        print("\n" + "─" * 60)
        print("UPDATED SUMMARY")
        print("─" * 60)
        print(f"Total top-level comments (all subreddits): {len(combined):,}")
        print(f"Target posts (bat_score>0):    {len(target_ids):,}")
        print(f"Posts with ≥1 surviving comment: {covered:,} "
              f"({covered/len(target_ids)*100:.1f}%)")


if __name__ == '__main__':
    main()

CLEAN + MERGE NEW SYSADMIN COMMENTS INTO MASTER FILE

Loaded /Users/nadia/Desktop/redditRun_june/comment_data/filtered_raw_sysadmin_comments_new.csv: 377,669 rows
Columns present: ['comment_id', 'post_id', 'created_utc', 'created_date', 'author', 'score', 'subreddit', 'body']

[WARN] Missing expected columns: ['id', 'link_id', 'parent_id', 'permalink']
       Update the column-mapping block below before proceeding,
       or the cleaning step will fail (needs at least: id, body, created_utc).
       Auto-applying rename: {'comment_id': 'id'}

[WARN] 2 of 377,669 rows have post_id NOT in bat_score>0 target set -- dropping these (they shouldn't be here).

Cleaning r/sysadmin comments (same logic as notebook)…
  Removed 64,836 duplicate comments
  Date window filter: kept 312,794 (removed 37)
  Removed 9,280 deleted/removed comments
  Removed 35,355 comments under 10 words
  Removed 0 comments with empty cleaned text

  r/sysadmin final cleaned count: 268,159

Existing master_comments_fil